# Notebook 08: Entity Resolution and Deduplication

## Why This Matters

Entity resolution - identifying when two records refer to the same real-world entity - is foundational to data quality at scale. 'Apple Inc.' vs 'Apple, Inc.' vs 'Apple Computer' are the same company. At scale (millions to billions of records), naive O(n^2) pairwise comparison is computationally impossible. Understanding blocking, similarity computation, and ML-based deduplication is essential for data engineering, platform, and ML roles.

In [ ]:
%pip install -q datasketch scikit-learn pandas numpy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import re
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
np.random.seed(42)
print("Ready.")

## 1. The Blocking Problem

Naive deduplication: compare every pair -> O(n^2). For n=1M records: 10^12 operations. Completely intractable.

**Blocking** generates candidate pairs (records that *might* match) before expensive similarity computation:
- **Exact blocking key**: records with same first 3 chars of last name + ZIP code
- **Sorted Neighborhood**: sort records, compare within sliding window
- **MinHash LSH**: probabilistic locality-sensitive hashing

**Key metrics**:
- **Pair completeness** (recall): what fraction of true matches are in candidate set?
- **Pair quality** (precision): what fraction of candidates are true matches?
- **Reduction ratio**: how much did we reduce the comparison space?

In [ ]:
def generate_catalog(n_products=500, duplicate_rate=0.15):
    base_names = ["iPhone 15 Pro", "Samsung Galaxy S24", "MacBook Pro 16", "Dell XPS 15",
                   "Sony WH-1000XM5", "Apple AirPods Pro", "Nike Air Max 270", "Adidas Ultraboost",
                   "Amazon Echo Dot", "Google Pixel 8"] * (n_products // 10 + 1)
    products = [{"id": i, "name": base_names[i], "is_duplicate_of": None} for i in range(n_products)]
    n_dups = int(n_products * duplicate_rate)
    for j in range(n_dups):
        orig_id = np.random.randint(0, n_products)
        orig_name = products[orig_id]["name"]
        noise_type = np.random.choice(["abbreviate", "punctuation", "typo", "extra_words"])
        if noise_type == "abbreviate": noisy = orig_name.replace("Pro", "Pr.").replace("Max", "Mx")
        elif noise_type == "punctuation": noisy = orig_name.replace(" ", "-").lower()
        elif noise_type == "typo": noisy = orig_name[:np.random.randint(1, len(orig_name))] + orig_name[np.random.randint(1, len(orig_name)):]
        else: noisy = orig_name + " (Renewed)" if np.random.rand() > 0.5 else "New " + orig_name
        products.append({"id": n_products + j, "name": noisy, "is_duplicate_of": orig_id})
    return pd.DataFrame(products)

catalog = generate_catalog(500)
print(f"Total: {len(catalog)}, Originals: {catalog.is_duplicate_of.isna().sum()}, Duplicates: {catalog.is_duplicate_of.notna().sum()}")
print("\nSample duplicates:")
for _, dup in catalog[catalog.is_duplicate_of.notna()].head(3).iterrows():
    orig = catalog[catalog.id == dup.is_duplicate_of].iloc[0]
    print(f"  Original: '{orig.name}' -> Duplicate: '{dup.name}'")

## 2. MinHash LSH for Scalable Blocking

**MinHash** efficiently estimates Jaccard similarity between sets. For text: represent each document as a set of character n-grams. MinHash produces a compact signature that preserves Jaccard similarity.

**LSH bands**: Band the MinHash signature. Records in the same band bucket become candidates. By choosing b (bands) and r (rows per band), you control the probability that similar documents become candidates:

`P(candidate) = 1 - (1 - s^r)^b`

where s is actual Jaccard similarity.

In [ ]:
from datasketch import MinHash, MinHashLSH

def text_to_shingles(text, k=3):
    text = re.sub(r"[^a-z0-9 ]", "", text.lower())
    return set(text[i:i+k] for i in range(len(text) - k + 1))

print("Building MinHash LSH index...")
lsh = MinHashLSH(threshold=0.3, num_perm=64)
minhashes = {}
for _, row in catalog.iterrows():
    m = MinHash(num_perm=64)
    for shingle in text_to_shingles(row.name):
        m.update(shingle.encode("utf8"))
    minhashes[row.id] = m
    lsh.insert(str(row.id), m)

candidate_pairs = set()
for record_id, mh in minhashes.items():
    for r in lsh.query(mh):
        pair = tuple(sorted([record_id, int(r)]))
        if pair[0] != pair[1]: candidate_pairs.add(pair)

true_pairs = set()
for _, row in catalog[catalog.is_duplicate_of.notna()].iterrows():
    true_pairs.add(tuple(sorted([row.id, row.is_duplicate_of])))

pair_completeness = len(true_pairs & candidate_pairs) / len(true_pairs) if true_pairs else 0
pair_quality = len(true_pairs & candidate_pairs) / len(candidate_pairs) if candidate_pairs else 0
reduction = 1 - len(candidate_pairs) / (len(catalog)*(len(catalog)-1)//2)

print(f"\n=== MinHash LSH Results ===")
print(f"Naive pairs: {len(catalog)*(len(catalog)-1)//2:,} | Candidate pairs: {len(candidate_pairs):,}")
print(f"Reduction ratio: {reduction:.3f} | Pair completeness (recall): {pair_completeness:.3f} | Pair quality (precision): {pair_quality:.3f}")

## 3. Pairwise Similarity and Supervised Deduplication

After blocking, score each candidate pair with richer similarity metrics:
- **Edit distance (Levenshtein)**: character-level edits; good for typos
- **Jaccard similarity**: set overlap; good for bag-of-words
- **Jaro-Winkler**: edit distance variant that weights prefix matches; good for names

Combine these into a feature vector, then train a binary classifier on labeled pairs.

In [ ]:
def levenshtein(s1, s2):
    if len(s1) < len(s2): return levenshtein(s2, s1)
    if not s2: return len(s1)
    prev = list(range(len(s2)+1))
    for i, c1 in enumerate(s1):
        curr = [i+1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j]+(c1!=c2), curr[j]+1, prev[j+1]+1))
        prev = curr
    return prev[-1]

def jaccard_sim(s1, s2, k=2):
    set1 = text_to_shingles(s1, k); set2 = text_to_shingles(s2, k)
    return len(set1 & set2) / len(set1 | set2) if (set1 or set2) else 0.0

def pair_features(name1, name2):
    n1, n2 = name1.lower(), name2.lower()
    edit_dist = levenshtein(n1, n2); max_len = max(len(n1), len(n2))
    return [1 - edit_dist/(max_len+1), jaccard_sim(n1,n2,2), jaccard_sim(n1,n2,3),
            min(len(n1),len(n2))/(max_len+1e-6), float(n1.split()[0]==n2.split()[0] if n1 and n2 else False)]

pair_feats, pair_labels = [], []
for pair in list(candidate_pairs)[:2000]:
    id1, id2 = pair
    row1 = catalog[catalog.id==id1].iloc[0]; row2 = catalog[catalog.id==id2].iloc[0]
    feats = pair_features(row1.name, row2.name)
    is_match = (row1.is_duplicate_of==id2 or row2.is_duplicate_of==id1)
    pair_feats.append(feats); pair_labels.append(int(is_match))

X_pairs, y_pairs = np.array(pair_feats), np.array(pair_labels)
print(f"Pair training set: {len(X_pairs)} pairs, {y_pairs.mean():.1%} matches")
X_tr, X_te, y_tr, y_te = train_test_split(X_pairs, y_pairs, test_size=0.2, stratify=y_pairs, random_state=42)
clf = LogisticRegression(class_weight="balanced", max_iter=500).fit(X_tr, y_tr)
from sklearn.metrics import classification_report
print(classification_report(y_te, clf.predict(X_te), target_names=["non-match","match"]))

## Summary

### Key Concepts

| Concept | Key Point |
|---|---|
| Blocking | Reduce O(n^2) to O(n log n) by generating only plausible candidate pairs |
| MinHash LSH | Probabilistic blocking using Jaccard similarity sketches |
| Pair completeness | Recall of blocking - what fraction of true matches are in candidates? |
| Edit distance | Character-level similarity for typo-robust matching |
| Supervised dedup | Train binary classifier on (pair features, is_match) labels |

### Common Pitfalls
- Running naive O(n^2) comparison on large datasets
- Optimizing only pair completeness (too many candidates -> expensive scoring)
- Using only lexical similarity for semantic entity resolution
